In [0]:
%python
# Databricks notebook source
from pyspark.sql.functions import current_timestamp, col

raw_data_path = "/Volumes/market_prices_db/bronze/landing_zone/wfp_market_food_prices.csv"
bronze_table_name = "market_prices_db.bronze.wfp_raw_prices"

# Leitura dos dados brutos do Volume
df_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(raw_data_path)

# Adição de metadados de linhagem (usando a coluna oculta _metadata permitida pelo Unity Catalog)
df_bronze = (df_raw
             .withColumn("ingestion_timestamp", current_timestamp())
             .withColumn("source_file", col("_metadata.file_path")))

# BRONZE: Inserção em modo Append para manter o histórico imutável
df_bronze.write.format("delta").mode("append").saveAsTable(bronze_table_name)